## Comprobación GPU

In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.current_device())
print(torch.cuda.get_device_name(0))


True
1
0
NVIDIA GeForce RTX 2060


## ResNet-50 Training

In [103]:
import os
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from pathlib import Path
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torch.nn as nn
from tqdm import tqdm  # Barra de progreso

# --- Configuración ---
CONFIG = {
    'data_dir': Path(r"C:\Users\alexa\Desktop\yolov4\Imagenes"),
    'ruta_guardado': r"C:\Users\alexa\Desktop\yolov4\models\resnet50_best.pth",
    'batch_size': 64,
    'learning_rate': 0.0002,
    'weight_decay': 0.0001,
    'epochs': 20,
    'patience': 10,
    'num_workers': 4,
    'pin_memory': True,
    'freeze_until_layer': 'layer4'  # None para no congelar, 'layer4' para congelar hasta layer3
}

# Asegurar que la GPU se usa si está disponible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando dispositivo:", device)

# Crear la carpeta si no existe
os.makedirs(os.path.dirname(CONFIG['ruta_guardado']), exist_ok=True)

# --- Transformaciones de Datos ---
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=20),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_valid = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Cargar datasets
train_dataset = ImageFolder(root=f"{CONFIG['data_dir']}/train", transform=transform_train)
valid_dataset = ImageFolder(root=f"{CONFIG['data_dir']}/valid", transform=transform_valid)

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=CONFIG['num_workers'], pin_memory=CONFIG['pin_memory'])
valid_loader = DataLoader(valid_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=CONFIG['num_workers'], pin_memory=CONFIG['pin_memory'])

# --- Modelo ResNet-50 ---
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

# Congelar capas base
if CONFIG['freeze_until_layer']:
    for name, param in model.named_parameters():
        if CONFIG['freeze_until_layer'] not in name and "fc" not in name:
            param.requires_grad = False

# Reemplazar la capa final
num_ftrs = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(num_ftrs, 38)
)
model = model.to(device)

# --- Configurar Entrenamiento ---
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay'])
scaler = torch.cuda.amp.GradScaler()
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.2, patience=3)

# --- Funciones de Entrenamiento y Validación ---
def train_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    running_loss = 0.0
    progress_bar = tqdm(loader, desc="Entrenando")
    for images, labels in progress_bar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()
        progress_bar.set_postfix({'loss': running_loss / (progress_bar.n + 1)})
    return running_loss / len(loader)

def validate_epoch(model, loader, criterion, device):
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        progress_bar = tqdm(loader, desc="Validando")
        for images, labels in progress_bar:
            images, labels = images.to(device), labels.to(device)
            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            progress_bar.set_postfix({'loss': val_loss / (progress_bar.n + 1), 'accuracy': 100 * correct / total})
    return val_loss / len(loader), 100 * correct / total

# --- Entrenamiento con Early Stopping ---
best_val_loss = float('inf')
epochs_no_improve = 0

for epoch in range(CONFIG['epochs']):
    train_loss = train_epoch(model, train_loader, criterion, optimizer, scaler, device)
    val_loss, accuracy = validate_epoch(model, valid_loader, criterion, device)

    print(f"Época {epoch+1}/{CONFIG['epochs']}, Pérdida de Entrenamiento: {train_loss:.4f}")
    print(f"✅ Validación - Pérdida: {val_loss:.4f}, Precisión: {accuracy:.2f}%")

    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), CONFIG['ruta_guardado'])
        print("💾 Mejor modelo guardado.")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= CONFIG['patience']:
            print(f"🛑 Deteniendo el entrenamiento anticipadamente después de {CONFIG['patience']} épocas sin mejora en la pérdida de validación.")
            break

print("Entrenamiento completado.")
print(f"💾 Mejor modelo guardado en: {CONFIG['ruta_guardado']}")


Usando dispositivo: cuda


C:\Users\alexa\AppData\Local\Temp\ipykernel_10844\2494912579.py:78: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
Entrenando:   0%|                                                                             | 0/1099 [00:00<?, ?it/s]C:\Users\alexa\AppData\Local\Temp\ipykernel_10844\2494912579.py:89: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Validando:   0%|                                                                               | 0/275 [00:00<?, ?it/s]C:\Users\alexa\AppData\Local\Temp\ipykernel_10844\2494912579.py:108: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Validando: 100%|█████████████████████████████████████████| 275/275 [00:38<00:

Época 1/20, Pérdida de Entrenamiento: 0.2410
✅ Validación - Pérdida: 0.0805, Precisión: 97.37%
💾 Mejor modelo guardado.


Validando: 100%|███████████████████████████████████████████| 275/275 [00:38<00:00,  7.07it/s, loss=0.0595, accuracy=98]


Época 2/20, Pérdida de Entrenamiento: 0.0779
✅ Validación - Pérdida: 0.0593, Precisión: 98.03%
💾 Mejor modelo guardado.


...

Época 18/20, Pérdida de Entrenamiento: 0.0076
✅ Validación - Pérdida: 0.0069, Precisión: 99.81%
💾 Mejor modelo guardado.


Validando: 100%|████████████████████████████████████████| 275/275 [00:39<00:00,  7.01it/s, loss=0.00654, accuracy=99.8]


Época 19/20, Pérdida de Entrenamiento: 0.0074
✅ Validación - Pérdida: 0.0065, Precisión: 99.80%
💾 Mejor modelo guardado.


Validando: 100%|████████████████████████████████████████| 275/275 [00:39<00:00,  7.03it/s, loss=0.00703, accuracy=99.8]

Época 20/20, Pérdida de Entrenamiento: 0.0073
✅ Validación - Pérdida: 0.0070, Precisión: 99.77%
Entrenamiento completado.
💾 Mejor modelo guardado en: C:\Users\alexa\Desktop\yolov4\models\resnet50_best.pth


## Optimización para raspberry pi 5

In [ ]:
 import torch
from torchvision.models import resnet50

model = resnet50()
model.load_state_dict(torch.load(r"C:\Users\alexa\Desktop\yolov4\models\resnet50_best.pth"))
model.eval()

traced = torch.jit.trace(model, torch.randn(1, 3, 224, 224))
traced.save("resnet50_traced.pt")

## Testing con imagenes de Internet

In [119]:
import torch
import torchvision.transforms as transforms
from PIL import Image
import os

# Asegúrate de que tu modelo esté definido igual que durante el entrenamiento
import torchvision.models as models
import torch.nn as nn

# Cargar el modelo entrenado (asegúrate de tener la ruta correcta)
model = models.resnet50(weights=None) # No cargar pesos preentrenados de ImageNet ahora
num_ftrs = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(num_ftrs, 38)
)
model.load_state_dict(torch.load("C:\\Users\\alexa\\Desktop\\yolov4\\models\\resnet50_best.pth", map_location=torch.device('cpu')))
model.eval()

# Definir las transformaciones (las mismas usadas para la validación)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Ruta al directorio de las imágenes a evaluar
image_dir = r"C:\Users\alexa\Desktop\yolov4\testing"

# Mapeo de índices de clase a nombres de clase (si tienes uno)
class_names = train_dataset.classes if 'train_dataset' in locals() else None

# Umbral de confianza para detectar imágenes fuera de distribución
umbral_confianza = 0.7

# Listar todos los archivos en el directorio
image_files = [f for f in os.listdir(image_dir) if os.path.isfile(os.path.join(image_dir, f))]

for image_file in image_files:
    image_path = os.path.join(image_dir, image_file)
    try:
        image = Image.open(image_path).convert("RGB")
        image_tensor = transform(image).unsqueeze(0)

        with torch.no_grad():
            outputs = model(image_tensor)
            probabilities = torch.nn.functional.softmax(outputs, dim=1)
            max_probability, predicted_class = torch.max(probabilities, 1)

        print(f"Archivo: {image_file}")
        print(f"Probabilidad máxima: {max_probability.item():.4f}")

        if max_probability.item() >= umbral_confianza:
            predicted_label = class_names[predicted_class.item()] if class_names else f"Clase {predicted_class.item()}"
            print(f"Predicción: {predicted_label} (Confianza: {max_probability.item():.4f})")
        else:
            print("Predicción: Imagen no reconocida con suficiente confianza (fuera de distribución).")
        print("-" * 30)

    except FileNotFoundError:
        print(f"Error: No se encontró el archivo {image_path}")
    except Exception as e:
        print(f"Error al procesar la imagen {image_file}: {e}")

Archivo: Apple_Black_rot_1.jpg
Probabilidad máxima: 0.4962
Predicción: Imagen no reconocida con suficiente confianza (fuera de distribución).
------------------------------
Archivo: Apple_Black_rot_2.jpg
Probabilidad máxima: 0.9506
Predicción: Apple_Black_rot (Confianza: 0.9506)
------------------------------
Archivo: Apple_Cedar_apple_rust_1.jpg
Probabilidad máxima: 0.9879
Predicción: Apple_Cedar_apple_rust (Confianza: 0.9879)
------------------------------
Archivo: Apple_Cedar_apple_rust_2.jpg
Probabilidad máxima: 0.4310
Predicción: Imagen no reconocida con suficiente confianza (fuera de distribución).
------------------------------
Archivo: Apple_Cedar_apple_rust_3.jpg
Probabilidad máxima: 0.8928
Predicción: Apple_Cedar_apple_rust (Confianza: 0.8928)
------------------------------
Archivo: audi_1.jpg
Probabilidad máxima: 0.3496
Predicción: Imagen no reconocida con suficiente confianza (fuera de distribución).
------------------------------
Archivo: balon_fut.jpg
Probabilidad máxima: